### 1. System Setup & Parameters
Initializes base interest rates, maximum risk ceilings, and the secure audit database.

In [16]:
import time
from datetime import datetime

# --- Enterprise B2B System Configurations ---
BASE_INTEREST_RATE = 4.0
MAX_POST_LOAN_LEVERAGE = 3.0  # Absolute ceiling for Total Debt / Equity
MIN_GOLDEN_RULE_RATIO = 0.5

# Initialize the secure audit database
audit_database = []

### 2. Financial Risk Engine
Calculates current balance sheet health and projects post-loan leverage risk.

In [17]:
def log_decision(company_name, status, details):
    """Saves corporate transactions to an audit trail."""
    record = {
        "timestamp": datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
        "company": company_name,
        "decision": status,
        "notes": details
    }
    audit_database.append(record)
    print(f"[AUDIT] Decision for {company_name} saved to secure database.")

def calculate_corporate_metrics(eigenkapital, fremdkapital, anlagevermoegen, loan_amount):
    """Calculates corporate metrics, projecting the impact of the new loan."""
    if eigenkapital <= 0:
        return {"equity_ratio": 0.0, "pre_leverage": 999.0, "post_leverage": 999.0, "golden_rule": 0.0}

    total_capital = eigenkapital + fremdkapital
    equity_ratio = eigenkapital / total_capital
    pre_leverage = fremdkapital / eigenkapital

    # Calculate the financial risk IF the loan is approved
    post_loan_debt = fremdkapital + loan_amount
    post_leverage = post_loan_debt / eigenkapital

    if anlagevermoegen > 0:
        golden_rule = eigenkapital / anlagevermoegen
    else:
        golden_rule = 1.0

    return {
        "equity_ratio": equity_ratio,
        "pre_leverage": pre_leverage,
        "post_leverage": post_leverage,
        "golden_rule": golden_rule
    }

def calculate_corporate_interest(equity_ratio, post_leverage):
    """Applies the risk formula, accounting for post-loan debt load."""
    risk_penalty = 15 * (1 - equity_ratio)**2
    leverage_penalty = post_leverage * 0.5
    final_rate = BASE_INTEREST_RATE + risk_penalty + leverage_penalty
    return round(final_rate, 2)

### 3. Decision Logic & Hybrid Oversight
Evaluates metrics to auto-approve safe loans, auto-deny high risk, or route borderline cases to a human underwriter.

In [18]:
def get_secure_input(prompt):
    """Loops until valid corporate financial data is entered."""
    while True:
        user_input = input(prompt).strip().replace(',', '')
        try:
            return float(user_input)
        except ValueError:
            print("[ERROR] Invalid format. Please enter numbers only.")

def collect_corporate_data():
    """Compiles secure balance sheet inputs."""
    print("\n--- NEW CORPORATE LOAN APPLICATION ---")
    return {
        "company": input("Company Name: ").strip(),
        "eigenkapital": get_secure_input("Eigenkapital / Equity (EUR): "),
        "fremdkapital": get_secure_input("Fremdkapital / Existing Debt (EUR): "),
        "anlagevermoegen": get_secure_input("Anlagevermoegen / Fixed Assets (EUR): "),
        "loan": get_secure_input("Requested Loan Amount (EUR): ")
    }

def evaluate_corporate_loan(data):
    """Evaluates corporate health and routes to algorithm or Human Underwriter."""
    comp = data["company"]
    metrics = calculate_corporate_metrics(data["eigenkapital"], data["fremdkapital"], data["anlagevermoegen"], data["loan"])
    rate = calculate_corporate_interest(metrics["equity_ratio"], metrics["post_leverage"])

    print(f"\n[SYSTEM ANALYSIS] Scanning Balance Sheet & Projecting Loan Impact for {comp}...")
    time.sleep(1.5)
    print(f"  > Pre-Loan Leverage: {metrics['pre_leverage']:.2f} | Projected Post-Loan Leverage: {metrics['post_leverage']:.2f}")
    print(f"  > Golden Rule: {metrics['golden_rule']:.2f} | Current Equity Ratio: {metrics['equity_ratio']*100:.1f}%")

    # 1. MICRO-LOAN (Auto-Approve)
    if data["eigenkapital"] > 0 and (data["loan"] / data["eigenkapital"]) < 0.05:
        log_decision(comp, "AUTO-APPROVED", "Micro-loan anomaly. Zero systemic risk.")
        return f"[APPROVED] Micro-loan anomaly. Terms: EUR {data['loan']:,.2f} @ {rate}%"

    # 2. HIGH RISK (Auto-Deny)
    elif metrics["post_leverage"] > MAX_POST_LOAN_LEVERAGE or metrics["golden_rule"] < MIN_GOLDEN_RULE_RATIO:
        log_decision(comp, "AUTO-DENIED", f"Projected Leverage ({metrics['post_leverage']:.2f}) exceeds bank limits.")
        return f"[DENIED] The requested loan amount creates an unsustainable debt burden."

    # 3. OPTIMAL HEALTH (Auto-Approve)
    elif metrics["post_leverage"] <= 1.5 and metrics["golden_rule"] >= 0.8:
        log_decision(comp, "AUTO-APPROVED", "Strong balance sheet. Loan absorption is optimal.")
        return f"[APPROVED] Optimal corporate health and safe loan sizing. Terms: EUR {data['loan']:,.2f} @ {rate}%"

    # 4. SYSTEM HALT (Human-in-the-Loop)
    else:
        print("\n[WARNING - SYSTEM HALT] Borderline risk detected. Loan substantially increases corporate leverage.")
        print("[MANUAL ROUTING] Routing to Corporate Underwriter for manual risk assessment...")

        while True:
            decision = input(f"Reviewing {comp} - Approve? (Y/N): ").strip().upper()
            if decision == 'Y':
                log_decision(comp, "HUMAN-APPROVED", "Executive override authorized after leverage review.")
                return f"[APPROVED] (Human Override). Terms: EUR {data['loan']:,.2f} @ {rate}%"
            elif decision == 'N':
                log_decision(comp, "HUMAN-DENIED", "Executive rejected due to excessive post-loan debt exposure.")
                return "[DENIED] (Human Override applied)."
            else:
                print("Please type 'Y' for Yes or 'N' for No.")

### 4. System Execution
Run the cell below to launch the interactive loan application terminal.

In [ ]:
def main():
    print("="*70)
    print(" ENTERPRISE B2B CORPORATE LOAN ORIGINATION SYSTEM")
    print("="*70)

    corp_data = collect_corporate_data()
    result = evaluate_corporate_loan(corp_data)

    print("\n" + "="*70)
    print(" FINAL CORPORATE DISPOSITION REPORT")
    print("="*70)
    print(result)
    print("="*70)

    print("\n--- Current Audit Database ---")
    if not audit_database:
        print("No records found.")
    for log in audit_database:
        print(f"[{log['timestamp']}] {log['company']} - {log['decision']}")

# Execute the application
if __name__ == "__main__":
    main()

 ENTERPRISE B2B CORPORATE LOAN ORIGINATION SYSTEM

--- NEW CORPORATE LOAN APPLICATION ---
